In [ ]:
from pathlib import Path
import sys

sys.path.insert(0, str(Path("../..").resolve()))

import matplotlib.pyplot as plt
import torch
from torch.utils.data import Subset
import ad_safe_lib as ad_safe

print("device:", ad_safe.DEVICE)
print("dataset:", ad_safe.DATA_DIR)

# Load prediction model once at notebook start so all demo cells use the same model.
TEACHER_PATH = ad_safe.AD_SAFE_RUNS_DIR / "2026-04-21-13-49-33-model.pt"
attack_model = ad_safe.load_model(TEACHER_PATH)
attack_model.eval()
print("Attack model:", ad_safe.get_model_display_name(attack_model))

# Enrichment Strategy Visualization

This notebook demonstrates enrichment strategies on a small sample subset and shows model predictions.
Row 0 is the source image, row 1 is the enriched image.
Titles display ground truth (`gt`) and model prediction (`pred`).

Model is loaded once in Cell 1 and passed explicitly to strategy visualizations.

## Load a small balanced subset from train

In [ ]:
N_SOURCES = 4  # 2 safe + 2 unsafe

train_ds = ad_safe.load_dataset("train")
class_names = ad_safe.get_dataset_classes(train_ds)
targets = ad_safe.get_dataset_targets(train_ds)

safe_idx   = [i for i, t in enumerate(targets) if t == 0][:N_SOURCES // 2]
unsafe_idx = [i for i, t in enumerate(targets) if t == 1][:N_SOURCES // 2]
sample_indices = safe_idx + unsafe_idx

subset = Subset(train_ds, sample_indices)
print(f"Source samples: {len(subset)}")
for i, idx in enumerate(sample_indices):
    _, label = train_ds[idx]
    print(f"  [{i}] global_idx={idx}  label={class_names[label]}")

## Helpers

In [ ]:
def tensor_to_hwc(t):
    return t.detach().cpu().permute(1, 2, 0).clamp(0, 1).numpy()


def predict_label_name(model, image_tensor):
    if model is None:
        return None
    model.eval()
    with torch.no_grad():
        logits = ad_safe.forward_logits(model, image_tensor.unsqueeze(0).to(ad_safe.DEVICE))
        pred_idx = int(logits.argmax(dim=1).item())
    return class_names[pred_idx]


class MinimalAdversarialStrategy(ad_safe.StrictInheritanceStrategy):
    """Single-sample minimal PGD flip strategy for notebook visualization."""

    def __init__(self, *, max_epsilon=0.05, num_steps=5, search_steps=8, refinement_steps=6):
        self.max_epsilon = float(max_epsilon)
        self.num_steps = int(num_steps)
        self.search_steps = int(search_steps)
        self.refinement_steps = int(refinement_steps)

    def run(self, subset, student_model, teacher_model):
        if student_model is None:
            raise ValueError("MinimalAdversarialStrategy requires student_model")

        student_model.eval()
        criterion = torch.nn.CrossEntropyLoss()
        results = []

        for source_pos in range(len(subset)):
            item = subset[source_pos]
            image, label = item[0], int(item[1])

            x = image.unsqueeze(0).to(ad_safe.DEVICE)
            y = torch.tensor([label], device=ad_safe.DEVICE)

            with torch.inference_mode():
                pred = int(ad_safe.forward_logits(student_model, x).argmax(dim=1).item())
            if pred != label:
                continue

            attack_result = ad_safe.generate_adversarial_perturbation(
                model=student_model,
                x_original=x,
                criterion=criterion,
                target_labels=y,
                strategy=ad_safe.MinimalFlipPgdStrategy(
                    max_epsilon=self.max_epsilon,
                    num_steps=self.num_steps,
                    search_steps=self.search_steps,
                    refinement_steps=self.refinement_steps,
                ),
            )

            if attack_result.attack_success:
                results.append((source_pos, attack_result.adversarial_tensor.squeeze(0).detach().cpu()))

        return results


def show_strategy_strip(title, strategy, subset, *, student_model=None, teacher_model=None):
    """Row 0 = source images, Row 1 = derived images. Shows gt and optional predicted labels."""
    n = len(subset)
    derived = strategy.run(subset, student_model, teacher_model)

    # one derived per source (first hit wins)
    src_to_derived = {}
    for src_pos, img_t in derived:
        src_to_derived.setdefault(src_pos, img_t)

    fig, axes = plt.subplots(2, n, figsize=(3 * n, 6))
    fig.suptitle(title, fontsize=13, fontweight="bold", y=1.01)

    for col in range(n):
        item = subset[col]
        src_img, src_label = item[0], item[1]
        label_str = class_names[src_label]
        src_pred = predict_label_name(student_model, src_img)

        axes[0, col].imshow(tensor_to_hwc(src_img))
        src_title = f"source [{col}]\ngt: {label_str}"
        if src_pred is not None:
            src_title += f"\npred: {src_pred}"
        axes[0, col].set_title(src_title, fontsize=9)
        axes[0, col].axis("off")

        d_img = src_to_derived.get(col)
        if d_img is not None:
            d_pred = predict_label_name(student_model, d_img)
            axes[1, col].imshow(tensor_to_hwc(d_img))
            d_title = f"{title}\n→ gt: {label_str}"
            if d_pred is not None:
                d_title += f"\npred: {d_pred}"
            axes[1, col].set_title(d_title, fontsize=9)
        else:
            axes[1, col].text(0.5, 0.5, "attack\nfailed", ha="center", va="center", fontsize=9)
            axes[1, col].set_title(f"{title}\n→ no success", fontsize=9)
        axes[1, col].axis("off")

    plt.tight_layout()
    plt.show()
    print(f"  {title}: {len(derived)} sample(s) derived from {n} source(s)")

## Geometric strategies

In [ ]:
show_strategy_strip("HorizontalFlip", ad_safe.HorizontalFlipStrategy(), subset, student_model=attack_model)

In [ ]:
show_strategy_strip("VerticalFlip", ad_safe.VerticalFlipStrategy(), subset, student_model=attack_model)

In [ ]:
show_strategy_strip("Rotate 90°", ad_safe.RotateStrategy(angles=(90,)), subset, student_model=attack_model)

In [ ]:
show_strategy_strip("Rotate 180°", ad_safe.RotateStrategy(angles=(180,)), subset, student_model=attack_model)

In [ ]:
show_strategy_strip("Rotate 270°", ad_safe.RotateStrategy(angles=(270,)), subset, student_model=attack_model)

In [ ]:
show_strategy_strip("Scale (0.9–1.1×)", ad_safe.ScaleStrategy(factor_min=0.9, factor_max=1.1), subset, student_model=attack_model)

In [ ]:
show_strategy_strip("GaussianBlur (σ 0.1–2.0)", ad_safe.GaussianBlurStrategy(), subset, student_model=attack_model)

In [ ]:
show_strategy_strip("Perspective (scale=0.2)", ad_safe.PerspectiveStrategy(distortion_scale=0.2), subset, student_model=attack_model)

In [ ]:
show_strategy_strip("Grayscale (3ch)", ad_safe.GrayscaleStrategy(), subset, student_model=attack_model)

## Adversarial strategy (minimal perturbation)

Model is already loaded in Cell 1 (`attack_model`).
This section uses `MinimalFlipPgdStrategy` to search for the smallest successful adversarial perturbation per sample.

In [ ]:
# Model is loaded in Cell 1. Re-run this cell only if you want to reload it.
print("Attack model:", ad_safe.get_model_display_name(attack_model))
print("Model path:", TEACHER_PATH)

In [ ]:
show_strategy_strip(
    "Adversarial MinimalFlip (max ε=0.05, steps=5)",
    MinimalAdversarialStrategy(max_epsilon=0.05, num_steps=5, search_steps=8, refinement_steps=6),
    subset,
    student_model=attack_model,
)

## Multi-phase enrichment job pipeline

Two chained enrichment jobs:
- **Job 0** — phases run in parallel on the original subset snapshot: `h_flip` + `rotate(90,180,270)`  
- **Job 1** — runs on the *output* of job 0 (which now includes the geometric augmentations): `adversarial`

All derived samples inherit the label of their respective source.

In [ ]:
enrichment_jobs = (
    ad_safe.EnrichmentJobSpec(phases=(
        ad_safe.EnrichmentPhaseSpec(strategy=ad_safe.HorizontalFlipStrategy()),
        ad_safe.EnrichmentPhaseSpec(strategy=ad_safe.RotateStrategy(angles=(90, 180, 270))),
    )),
    ad_safe.EnrichmentJobSpec(phases=(
        ad_safe.EnrichmentPhaseSpec(
            strategy=MinimalAdversarialStrategy(
                max_epsilon=0.05,
                num_steps=5,
                search_steps=8,
                refinement_steps=6,
            )
        ),
    )),
)

enriched = ad_safe.run_enrichment_jobs(
    jobs=enrichment_jobs,
    subset=subset,
    student_model=attack_model,
    teacher_model=None,
    teacher_logits=None,
)
n_orig = len(subset)
print(f"Original subset : {n_orig} samples")
print(f"Enriched dataset: {len(enriched)} samples")
print(f"Added            : {len(enriched.extra_samples)} samples")

In [ ]:
# Display all samples with optional predictions from attack_model
n_total = len(enriched)
cols = min(4, n_total)
rows = (n_total + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows))
axes_flat = axes.flatten() if hasattr(axes, 'flatten') else [axes]
pred_model = attack_model

for i in range(n_total):
    item = enriched[i]
    img, label = item[0], item[1]
    label_str = class_names[label]
    pred_str = predict_label_name(pred_model, img)

    if i < n_orig:
        title = f"orig [{i}]\ngt: {label_str}"
        color = "black"
    else:
        title = f"enriched #{i - n_orig}\n→ gt: {label_str}"
        color = "darkblue"
    if pred_str is not None:
        title += f"\npred: {pred_str}"

    axes_flat[i].imshow(tensor_to_hwc(img))
    axes_flat[i].set_title(title, fontsize=8, color=color)
    axes_flat[i].axis("off")

for i in range(n_total, len(axes_flat)):
    axes_flat[i].axis("off")

fig.suptitle(
    f"Pipeline output: {n_orig} original + {n_total - n_orig} enriched = {n_total} total\n"
    "black = original   blue = enriched",
    fontsize=11,
)
plt.tight_layout()
plt.show()

## Label inheritance check

Every derived sample must carry the same label as its source.

In [ ]:
source_labels = {i: int(subset[i][1]) for i in range(len(subset))}
print("Source labels:", {i: class_names[v] for i, v in source_labels.items()})
print()
print("Extra samples:")
for i, (_, label, *_rest) in enumerate(enriched.extra_samples):
    print(f"  extra[{i:02d}]: {class_names[label]}")

print()
print("✓ All derived samples inherit their source label (strict inheritance).")